# Feature Engineering

## Objective

The objective of this notebook is to create new business features from the cleaned Superstore dataset. These engineered features improve business analysis and prepare the dataset for customer segmentation, machine learning, and dashboard development.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# Load cleaned dataset
df = pd.read_csv("../data/processed/clean_superstore.csv")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Order Month Name,Order Quarter
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,November,4
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,November,4
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,June,2
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,October,4
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,October,4


# Dataset Overview

Load the cleaned dataset generated in the previous phase and verify that it is ready for feature engineering.

In [3]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (9994, 25)
<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Row ID            9994 non-null   int64  
 1   Order ID          9994 non-null   str    
 2   Order Date        9994 non-null   str    
 3   Ship Date         9994 non-null   str    
 4   Ship Mode         9994 non-null   str    
 5   Customer ID       9994 non-null   str    
 6   Customer Name     9994 non-null   str    
 7   Segment           9994 non-null   str    
 8   Country           9994 non-null   str    
 9   City              9994 non-null   str    
 10  State             9994 non-null   str    
 11  Postal Code       9994 non-null   int64  
 12  Region            9994 non-null   str    
 13  Product ID        9994 non-null   str    
 14  Category          9994 non-null   str    
 15  Sub-Category      9994 non-null   str    
 16  Product Name      9994 non-

# Feature 1 – Shipping Duration

Calculate the number of days taken to deliver each order.

In [4]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

df["Shipping Duration"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

df[["Order Date", "Ship Date", "Shipping Duration"]].head()

,Order Date,Ship Date,Shipping Duration
0,2016-11-08,2016-11-11,3
1,2016-11-08,2016-11-11,3
2,2016-06-12,2016-06-16,4
3,2015-10-11,2015-10-18,7
4,2015-10-11,2015-10-18,7


## Business Insight

Shipping Duration measures delivery efficiency. Lower shipping durations indicate faster order fulfillment and better customer service.

# Feature 2 – Profit Margin

In [5]:
df["Profit Margin"] = (
    df["Profit"] / df["Sales"]
) * 100

df[["Sales", "Profit", "Profit Margin"]].head()

,Sales,Profit,Profit Margin
0,261.9600,41.9136,16.00
1,731.9400,219.5820,30.00
2,14.6200,6.8714,47.00
3,957.5775,-383.0310,-40.00
4,22.3680,2.5164,11.25


## Business Insight

Profit Margin measures how much profit is earned for every dollar of sales. Higher values indicate more profitable transactions.

# Feature 3 – Sales Per Unit

In [6]:
df["Sales Per Unit"] = (
    df["Sales"] / df["Quantity"]
)

df[["Sales", "Quantity", "Sales Per Unit"]].head()

,Sales,Quantity,Sales Per Unit
0,261.9600,2,130.9800
1,731.9400,3,243.9800
2,14.6200,2,7.3100
3,957.5775,5,191.5155
4,22.3680,2,11.1840


## Business Insight

Sales Per Unit allows comparison of product prices regardless of order quantity.

# Feature 4 – Discount Category

In [7]:
df["Discount Category"] = pd.cut(
    df["Discount"],
    bins=[-0.01, 0, 0.2, 0.5, 1],
    labels=[
        "No Discount",
        "Low",
        "Medium",
        "High"
    ]
)

df["Discount Category"].value_counts()

Discount Category
No Discount    4798
Low            3803
High            856
Medium          537
Name: count, dtype: int64

## Business Insight

Grouping discounts into categories makes business analysis easier and provides better features for machine learning.

# Feature 5 – Order Value Category

In [8]:
df["Order Value Category"] = pd.qcut(
    df["Sales"],
    q=3,
    labels=[
        "Low Value",
        "Medium Value",
        "High Value"
    ]
)

df["Order Value Category"].value_counts()

Order Value Category
Medium Value    3333
Low Value       3332
High Value      3329
Name: count, dtype: int64

## Business Insight

Orders are categorized into Low, Medium, and High Value groups for customer and sales analysis.

# Feature 6 – Weekend Orders

In [9]:
df["Order Day"] = df["Order Date"].dt.day_name()

df["Weekend Order"] = df["Order Day"].isin(
    ["Saturday", "Sunday"]
)

df[["Order Day", "Weekend Order"]].head()

,Order Day,Weekend Order
0,Tuesday,False
1,Tuesday,False
2,Sunday,True
3,Sunday,True
4,Sunday,True


## Business Insight

Weekend ordering patterns can help businesses optimize promotions and staffing.

# Feature 7 – Season

In [10]:
season_map = {
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Spring",
    6: "Summer",
    7: "Summer",
    8: "Summer",
    9: "Autumn",
    10: "Autumn",
    11: "Autumn"
}

df["Season"] = df["Order Month"].map(season_map)

df["Season"].value_counts()

Season
Autumn    3673
Summer    2133
Spring    2099
Winter    2089
Name: count, dtype: int64

## Business Insight

Seasonal information helps identify customer purchasing behavior across different times of the year.

# Feature 8 – Customer Lifetime Sales

In [11]:
customer_sales = (
    df.groupby("Customer ID")["Sales"]
      .transform("sum")
)

df["Customer Lifetime Sales"] = customer_sales

df[["Customer ID", "Customer Lifetime Sales"]].head()

,Customer ID,Customer Lifetime Sales
0,CG-12520,1148.7800
1,CG-12520,1148.7800
2,DV-13045,1119.4830
3,SO-20335,2602.5755
4,SO-20335,2602.5755


## Business Insight

Customer Lifetime Sales identifies customers who contribute the most revenue over time.

# Feature 9 – Customer Lifetime Profit

In [12]:
customer_profit = (
    df.groupby("Customer ID")["Profit"]
      .transform("sum")
)

df["Customer Lifetime Profit"] = customer_profit

df[["Customer ID", "Customer Lifetime Profit"]].head()

,Customer ID,Customer Lifetime Profit
0,CG-12520,169.9344
1,CG-12520,169.9344
2,DV-13045,-427.1840
3,SO-20335,-81.0858
4,SO-20335,-81.0858


## Business Insight

This feature identifies the most profitable customers and supports customer retention strategies.

# Save Feature Engineered Dataset

In [13]:
df.to_csv(
    "../data/processed/feature_engineered_superstore.csv",
    index=False
)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!


# Notebook Summary

## Features Created

- Shipping Duration
- Profit Margin
- Sales Per Unit
- Discount Category
- Order Value Category
- Weekend Order
- Season
- Customer Lifetime Sales
- Customer Lifetime Profit

## Benefits

These engineered features improve business analysis and prepare the dataset for customer segmentation, machine learning, and dashboard development in the upcoming phases.